# Training data chip creation
something descriptive about 300x300 dataset

In [1]:
!rm -rf ~/.local/lib/python*

In [2]:
!pip install rioxarray geopandas

Defaulting to user installation because normal site-packages is not writeable
  Using cached rioxarray-0.22.0-py3-none-any.whl.metadata (5.4 kB)
  Using cached xarray-2026.4.0-py3-none-any.whl.metadata (12 kB)
  Using cached pyogrio-0.12.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.9 kB)
Using cached rioxarray-0.22.0-py3-none-any.whl (72 kB)
Using cached pyogrio-0.12.1-cp312-cp312-manylinux_2_28_x86_64.whl (32.5 MB)
Using cached xarray-2026.4.0-py3-none-any.whl (1.4 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [rioxarray]/3 [rioxarray]


In [3]:
import json
import logging
import os
import sys
import time
import shutil
from glob import glob
from pathlib import Path
from contextlib import redirect_stdout, nullcontext
from functools import partial
from collections import Counter
from io import StringIO
import multiprocessing as mp
from multiprocessing import Pool, cpu_count, get_logger

import geopandas as gpd
import numpy as np
import rasterio
import xarray as xr
import rioxarray as rxr
from rasterio.enums import Resampling
from rasterio.crs import CRS
from tqdm import tqdm

try:
    import psutil
    HAS_PSUTIL = True
except ImportError:
    HAS_PSUTIL = False

In [ ]:
repo_dir = "lfm"

try:
    if not os.path.exists(repo_dir):
        result = subprocess.run(
            ["git", "clone", f"https://github.com/nasa-nccs-hpda/{repo_dir}"],
            check=True,
            capture_output=True,
            text=True
        )
        print(f"Successfully cloned repository to {repo_dir}/")
    else:
        result = subprocess.run(
            ["git", "-C", repo_dir, "pull"],
            check=True,
            capture_output=True,
            text=True
        )
        print(f"Successfully pulled latest changes in {repo_dir}/")
except (subprocess.CalledProcessError, FileNotFoundError) as e:
    print("Git subprocess call failed (known issue in Jupyter notebooks)")
    print("\nPlease run this command in your terminal:")
    print(f"  git clone https://github.com/nasa-nccs-hpda/{repo_dir}")

repoPath = Path(os.getcwd()) / repo_dir
sys.path.append(str(repoPath.parent))

In [ ]:
from lfm.model.Pipeline import Pipeline
from lfm.model.chip_making.chip_utils import run_pipeline_for_sample, process_train_sample
from lfm.model.chip_making.chip_constants import PROJECT_DIR, GPKG_PATH

# Load training geometries into GeoDataFrame

In [ ]:
train_gdf_path = GPKG_PATH
train_gdf = gpd.read_file(GPKG_PATH)
print(f"Loaded train gdf with {len(train_gdf)} entries.")